In [46]:
import os
import time
from pathlib import Path
import torch
import torch.nn as nn
from torchvision import transforms, models
from torchvision.models import DenseNet121_Weights, ResNet50_Weights
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights
from PIL import Image
import numpy as np

In [47]:
# Configuration
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# Paths
trained_models_dir = Path("../trained_models")
image_path = Path("../sample_image_begnign.jpg")

# Models to evaluate (exclude multitask_fl_model.pth)
model_files = [
    "deeplabv3_resnet50_best.pt",
    "densenet121_best.pt", 
    "densenet50_best.pt",
    "resnet50_best.pt"
]

Using device: mps


In [48]:
# Helper function to build DenseNet50 with custom config
def build_densenet50(num_classes=7):
    """Build DenseNet-50 with growth rate 32."""
    model = models.DenseNet(
        growth_rate=32,
        block_config=(6, 12, 24, 16),  # DenseNet-121 layers
        num_init_features=64,
        num_classes=num_classes
    )
    return model

In [49]:
# Load models function
def load_model(model_name, model_path, device):
    """Load a model from checkpoint based on its name."""
    checkpoint = torch.load(model_path, map_location=device)
    
    # Extract state dict - handle different checkpoint formats
    if 'model' in checkpoint:
        state_dict = checkpoint['model']
    elif 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict']
    elif 'state_dict' in checkpoint:
        state_dict = checkpoint['state_dict']
    else:
        # Checkpoint might be the state_dict directly
        state_dict = checkpoint
    
    if model_name.startswith("deeplabv3"):
        # Segmentation model - initialize with num_classes
        model = deeplabv3_resnet50(weights=None, num_classes=2)
        # Load weights with strict=False to ignore aux_classifier mismatches
        model.load_state_dict(state_dict, strict=False)
        
    elif model_name.startswith("densenet50"):
        # DenseNet50 classification model - custom block config (4, 8, 12, 16)
        model = models.DenseNet(
            growth_rate=32,
            block_config=(4, 8, 12, 16),
            num_init_features=64,
            bn_size=4,
            drop_rate=0.0,
            num_classes=1000,  # Default number of classes
            memory_efficient=False,
        )
        # Replace classifier with Sequential (Dropout + Linear) to match baseline
        model.classifier = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(model.classifier.in_features, 7)
        )
        model.load_state_dict(state_dict)
        
    elif model_name.startswith("densenet121"):
        # DenseNet121 classification model - match the baseline architecture
        model = models.densenet121(weights=None)
        model.classifier = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(model.classifier.in_features, 7)
        )
        model.load_state_dict(state_dict)
        
    elif model_name.startswith("resnet50"):
        # ResNet50 classification model - match the baseline architecture
        model = models.resnet50(weights=None)
        model.fc = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(model.fc.in_features, 7)
        )
        model.load_state_dict(state_dict)
    
    else:
        raise ValueError(f"Unknown model type: {model_name}")
    
    model = model.to(device)
    model.eval()
    return model

In [50]:
# Preprocessing transforms
def get_transform(model_name):
    """Get appropriate preprocessing transforms for each model type."""
    if model_name.startswith("deeplabv3"):
        # Segmentation model - use DeepLabV3 preprocessing
        weights = DeepLabV3_ResNet50_Weights.DEFAULT
        mean = tuple(float(m) for m in weights.meta.get("mean", (0.485, 0.456, 0.406)))
        std = tuple(float(s) for s in weights.meta.get("std", (0.229, 0.224, 0.225)))
        return transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std)
        ])
    else:
        # Classification models - use standard ImageNet preprocessing
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

In [51]:
# Load and preprocess image
def load_image(image_path, transform):
    """Load and preprocess an image."""
    image = Image.open(image_path).convert("RGB")
    return transform(image).unsqueeze(0)  # Add batch dimension

In [52]:
# Inference time measurement
def measure_inference_time(model, input_tensor, device, warmup_runs=10, num_runs=100):
    """Measure the average inference time of a model.
    
    Args:
        model: The PyTorch model
        input_tensor: Preprocessed input tensor
        device: Device to run inference on
        warmup_runs: Number of warmup iterations (not counted)
        num_runs: Number of timed inference runs
        
    Returns:
        Average inference time in milliseconds
    """
    input_tensor = input_tensor.to(device)
    
    # Warmup runs
    with torch.no_grad():
        for _ in range(warmup_runs):
            _ = model(input_tensor)
            if device.type == "cuda":
                torch.cuda.synchronize()
            elif device.type == "mps":
                torch.mps.synchronize()
    
    # Timed runs
    times = []
    with torch.no_grad():
        for _ in range(num_runs):
            start = time.time()
            _ = model(input_tensor)
            
            # Ensure computation is complete
            if device.type == "cuda":
                torch.cuda.synchronize()
            elif device.type == "mps":
                torch.mps.synchronize()
            
            end = time.time()
            times.append((end - start) * 1000)  # Convert to milliseconds
    
    return np.mean(times), np.std(times)

In [53]:
# Main evaluation loop
print("="*60)
print("MODEL INFERENCE TIME EVALUATION")
print("="*60)
print(f"\nEvaluation image: {image_path}")
print(f"Device: {device}")
print("\nLoading models and measuring inference time...\n")

results = []

for model_file in model_files:
    model_path = trained_models_dir / model_file
    
    if not model_path.exists():
        print(f"⚠️  Model not found: {model_file}")
        continue
    
    print(f"📊 {model_file}")
    print("-" * 60)
    
    try:
        # Load model
        model_name = model_file.replace("_best.pt", "")
        print(f"   Loading model...")
        model = load_model(model_name, model_path, device)
        
        # Get appropriate transform
        transform = get_transform(model_name)
        
        # Load and preprocess image
        print(f"   Preprocessing image...")
        input_tensor = load_image(image_path, transform)
        
        # Measure inference time
        print(f"   Measuring inference time (100 runs)...")
        avg_time, std_time = measure_inference_time(model, input_tensor, device)
        
        print(f"   ✅ Average inference time: {avg_time:.3f} ± {std_time:.3f} ms")
        print(f"   ✅ FPS: {1000/avg_time:.2f}\n")
        
        results.append({
            "model": model_file,
            "avg_time_ms": avg_time,
            "std_time_ms": std_time,
            "fps": 1000/avg_time
        })
        
    except Exception as e:
        print(f"   ❌ Error: {str(e)}\n")
        continue

print("="*60)
print("SUMMARY")
print("="*60)

MODEL INFERENCE TIME EVALUATION

Evaluation image: ../sample_image_begnign.jpg
Device: mps

Loading models and measuring inference time...

📊 deeplabv3_resnet50_best.pt
------------------------------------------------------------
   Loading model...
   Preprocessing image...
   Measuring inference time (100 runs)...
   ✅ Average inference time: 64.565 ± 1.359 ms
   ✅ FPS: 15.49

📊 densenet121_best.pt
------------------------------------------------------------
   Loading model...
   Preprocessing image...
   Measuring inference time (100 runs)...
   ✅ Average inference time: 16.939 ± 0.279 ms
   ✅ FPS: 59.04

📊 densenet50_best.pt
------------------------------------------------------------
   Loading model...
   Preprocessing image...
   Measuring inference time (100 runs)...
   ✅ Average inference time: 11.362 ± 0.254 ms
   ✅ FPS: 88.01

📊 resnet50_best.pt
------------------------------------------------------------
   Loading model...
   Preprocessing image...
   Measuring inference 

In [54]:
# Display results in a formatted table
if results:
    # Sort by inference time
    results_sorted = sorted(results, key=lambda x: x["avg_time_ms"])
    
    print(f"\n{'Model':<30} {'Avg Time (ms)':<15} {'Std (ms)':<15} {'FPS':<10}")
    print("-" * 70)
    
    for r in results_sorted:
        print(f"{r['model']:<30} {r['avg_time_ms']:>14.3f} {r['std_time_ms']:>14.3f} {r['fps']:>9.2f}")
    
    print("\n" + "="*60)
    print(f"Fastest model: {results_sorted[0]['model']} ({results_sorted[0]['avg_time_ms']:.3f} ms)")
    print(f"Slowest model: {results_sorted[-1]['model']} ({results_sorted[-1]['avg_time_ms']:.3f} ms)")
    print("="*60)
else:
    print("\n⚠️  No models were successfully evaluated.")


Model                          Avg Time (ms)   Std (ms)        FPS       
----------------------------------------------------------------------
resnet50_best.pt                       10.049          0.066     99.51
densenet50_best.pt                     11.362          0.254     88.01
densenet121_best.pt                    16.939          0.279     59.04
deeplabv3_resnet50_best.pt             64.565          1.359     15.49

Fastest model: resnet50_best.pt (10.049 ms)
Slowest model: deeplabv3_resnet50_best.pt (64.565 ms)


In [56]:
# Create comprehensive table with parameters and inference time
import pandas as pd

# Model specifications with inference time data
model_data = {
    'resnet50_best.pt': {'avg_time_ms': 10.049, 'std_ms': 0.066, 'fps': 99.51},
    'densenet50_best.pt': {'avg_time_ms': 11.362, 'std_ms': 0.254, 'fps': 88.01},
    'densenet121_best.pt': {'avg_time_ms': 16.939, 'std_ms': 0.279, 'fps': 59.04},
    'deeplabv3_resnet50_best.pt': {'avg_time_ms': 64.565, 'std_ms': 1.359, 'fps': 15.49}
}

# Calculate parameters for each model
model_params = {}
for model_file in model_data.keys():
    model_path = trained_models_dir / model_file
    model_name = model_file.replace("_best.pt", "")
    
    try:
        # Load model to count parameters
        model = load_model(model_name, model_path, device)
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        
        model_params[model_file] = {
            'total_params': total_params,
            'trainable_params': trainable_params
        }
        
        # Clean up
        del model
        
    except Exception as e:
        print(f"Error loading {model_file}: {e}")
        model_params[model_file] = {
            'total_params': 0,
            'trainable_params': 0
        }

# Create comprehensive dataframe
data_rows = []
for model_file in model_data.keys():
    row = {
        'Model': model_file.replace('_best.pt', ''),
        'Total Parameters': f"{model_params[model_file]['total_params']:,}",
        'Avg Time (ms)': f"{model_data[model_file]['avg_time_ms']:.3f}"
    }
    data_rows.append(row)

df_results = pd.DataFrame(data_rows)

# Sort by inference time
df_results = df_results.sort_values('Avg Time (ms)', key=lambda x: x.str.replace(',', '').astype(float))

print("\n" + "="*100)
print("MODEL COMPARISON: PARAMETERS AND INFERENCE TIME")
print("="*100)
print(df_results.to_string(index=False))
print("="*100)


MODEL COMPARISON: PARAMETERS AND INFERENCE TIME
             Model Total Parameters Avg Time (ms)
          resnet50       23,522,375        10.049
        densenet50        3,646,143        11.362
       densenet121        6,961,031        16.939
deeplabv3_resnet50       39,633,986        64.565
